# OpenAI Agents SDK — Practical Guide

**Philosophy:** Minimal, explicit, and close to the API. Designed to be simple and educational rather than a heavy production framework.

---

### Three Core Concepts

| Concept | What it means |
|---------|---------------|
| **Agents** | An LLM with a name, instructions, and optional tools or handoffs |
| **Handoffs** | How agents permanently transfer control to another agent |
| **Guardrails** | Safety checks applied to agent input and output |

## Quick Start

Three steps: **Create → Trace → Run**

### 1. Install & Import
```python
# pip install openai-agents
from dotenv import load_dotenv
from agents import Agent, Runner, trace

load_dotenv(override=True)
```

### 2. Create an Agent
```python
agent = Agent(
    name='Jokester',
    instructions='You are a joke teller. Keep jokes short and funny.',
    model='gpt-4o-mini'
)
```

### 3. Run with Tracing
```python
async def tell_joke():
    with trace('Telling a joke'):          # logs everything to OpenAI platform
        result = await Runner.run(agent, 'Tell a joke about AI agents')
        print(result.final_output)

await tell_joke()                          # in Jupyter; use asyncio.run() elsewhere
```

> **Traces dashboard:** https://platform.openai.com/logs?api=traces — shows all LLM calls, tool usage, costs, and timing per run.

## Agent Workflows — Sequential Pipelines

Chain agents so each one performs a specialized step.

```python
from agents import Agent, Runner

researcher = Agent(name="Researcher", instructions="Research topics and gather key facts.")
writer     = Agent(name="Writer",     instructions="Write engaging content from research notes.")
editor     = Agent(name="Editor",     instructions="Edit for grammar and clarity.")

async def content_pipeline(topic: str) -> str:
    research = await Runner.run(researcher, f"Research: {topic}")
    draft    = await Runner.run(writer,     research.final_output)
    final    = await Runner.run(editor,     draft.final_output)
    return final.final_output
```

Each agent runs independently — you orchestrate the sequence in Python.

---

## Custom Tools

Give an agent Python functions it can call autonomously.

```python
from agents import Agent, Runner

def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

def get_stock_price(symbol: str) -> str:
    """Get current stock price."""
    return f"{symbol}: $150.00"

research_agent = Agent(
    name="Research Assistant",
    instructions="Help users research topics. Use tools when needed.",
    tools=[search_web, get_stock_price]
)

result = await Runner.run(research_agent, "What's the current price of AAPL?")
# Agent calls get_stock_price("AAPL") automatically, then forms its reply
```

**How it works:** Agent decides which tool to call → SDK executes the function → Agent uses the return value to compose its response.

## Handoffs — Agent-to-Agent Delegation

Handoffs let one agent permanently pass control to a specialist agent.

```python
from agents import Agent, Runner

billing_agent   = Agent(name="Billing",   instructions="Handle payment and billing issues.")
technical_agent = Agent(name="Technical", instructions="Handle technical support.")

triage_agent = Agent(
    name="Triage",
    instructions="Route users to the right specialist: Billing or Technical.",
    handoffs=[billing_agent, technical_agent]
)

result = await Runner.run(triage_agent, "I have a technical problem with my account.")
# Triage agent hands off to Technical agent; Technical agent takes over the conversation
```

## Tools vs Handoffs

The most important distinction in the SDK.

| | Tools | Handoffs |
|---|---|---|
| **Control after call** | Returns to the calling agent | Passes permanently to the new agent |
| **Analogy** | Calling a function | Transferring a phone call |
| **Use when** | You need data or an action, then want to continue | A specialist should own the rest of the conversation |

### Combined Example — Customer Service

```python
from agents import Agent, Runner

# --- Tools: return data, control comes back ---
def get_order_status(order_id: str) -> str:
    """Look up an order in the database."""
    return f"Order {order_id}: Shipped, arriving tomorrow."

# --- Handoffs: transfer control permanently ---
billing_agent   = Agent(name="Billing",   instructions="Handle billing disputes.")
technical_agent = Agent(name="Technical", instructions="Handle technical support.")

main_agent = Agent(
    name="Customer Service",
    instructions="""
    Help customers.
    - Use get_order_status to look up orders.
    - Hand off billing issues to Billing agent.
    - Hand off technical problems to Technical agent.
    """,
    tools=[get_order_status],                    # control returns here
    handoffs=[billing_agent, technical_agent]    # control passes away
)
```

**Scenario A — uses tool:**
```
User:  "What's the status of order #123?"
Agent: calls get_order_status("123") → "Your order ships tomorrow."
```

**Scenario B — uses handoff:**
```
User:  "I was overcharged on my bill."
Agent: hands off to Billing agent → Billing agent takes over.
```

## Selecting Models

Each agent in a workflow can use a different model — mix for cost/quality balance.

```python
planner   = Agent(name="Planner",   model="gpt-4o",      instructions="Create detailed plans.")
executor  = Agent(name="Executor",  model="gpt-4o-mini", instructions="Execute steps quickly.")
summarizer = Agent(name="Summarizer", model="gpt-4o-mini", instructions="Summarize results.")
```

| Model | Quality | Speed | Cost | Best for |
|-------|---------|-------|------|----------|
| `gpt-4o` | High | Fast | $$$ | Complex reasoning, planning, production |
| `gpt-4o-mini` | Good | Faster | $$ | Most tasks — best value |
| `gpt-3.5-turbo` | Decent | Fastest | $ | Simple, high-volume tasks |

**Strategy:** Use `gpt-4o` for planning and decision-making steps; use `gpt-4o-mini` for execution, summarization, and guardrails.

## Structured Outputs with Pydantic

Force the agent to return a typed object instead of free-form text — useful for building reliable, parseable pipelines.

```python
from pydantic import BaseModel
from enum import Enum
from agents import Agent, Runner

class Priority(str, Enum):
    LOW    = "low"
    MEDIUM = "medium"
    HIGH   = "high"

class Task(BaseModel):
    title:           str
    priority:        Priority
    estimated_hours: int
    dependencies:    list[str]

task_analyzer = Agent(
    name="Task Analyzer",
    instructions="Analyze project tasks and return structured data.",
    output_type=Task
)

result = await Runner.run(
    task_analyzer,
    "Build a login system with OAuth. It's urgent, ~8 hours. Depends on database setup."
)

task = result.final_output_as(Task)
print(task.priority)          # Priority.HIGH
print(task.estimated_hours)   # 8
print(task.dependencies)      # ["database setup"]
```

**Benefits:** type safety, automatic Pydantic validation, IDE autocomplete, no manual JSON parsing.

## Guardrails

Guardrails are safety checks on what enters and exits your agent pipeline. They can be plain functions or full agents themselves.

> **Limitation:** Guardrails apply only to the **first agent's input** and the **last agent's output** — not mid-workflow.

```python
from pydantic import BaseModel
from agents import Agent, Runner
import logging

class SafetyCheck(BaseModel):
    is_safe:  bool
    category: str   # "safe" | "inappropriate" | "dangerous"
    reason:   str

input_guard = Agent(
    name="Input Safety",
    instructions="Decide if this user request is safe and appropriate.",
    output_type=SafetyCheck
)

output_guard = Agent(
    name="Output Safety",
    instructions="Decide if this response contains unsafe or sensitive content.",
    output_type=SafetyCheck
)

assistant = Agent(
    name="Assistant",
    instructions="Help users with their questions."
)

async def guarded_run(user_input: str) -> str:
    # Input guardrail
    check = await Runner.run(input_guard, user_input)
    if not check.final_output_as(SafetyCheck).is_safe:
        logging.warning(f"Blocked input: {user_input}")
        return "I cannot process that request."

    # Main agent
    result = await Runner.run(assistant, user_input)

    # Output guardrail
    check = await Runner.run(output_guard, result.final_output)
    if not check.final_output_as(SafetyCheck).is_safe:
        logging.warning(f"Blocked output: {result.final_output}")
        return "I cannot provide that response."

    return result.final_output
```

**Best practices:**
- Use a cheap, fast model for guardrail agents to keep latency low
- Log every rejection — review false positives and tune prompts
- Give users clear, helpful rejection messages

## OpenAI Hosted Tools

Pre-built tools maintained by OpenAI — no implementation required, just add to `tools`.

---

### WebSearchTool

Lets the agent search the web for real-time information beyond its training cutoff.

```python
from agents import Agent, Runner
from agents.tools import WebSearchTool

agent = Agent(
    name="Research Assistant",
    instructions="Find current information using web search.",
    tools=[WebSearchTool()]
)

result = await Runner.run(agent, "Latest AI agent developments in 2025?")
```

---

### FileSearchTool

Semantic search over documents uploaded to an OpenAI Vector Store — automatic RAG with no extra setup.

```python
from openai import OpenAI
from agents import Agent, Runner
from agents.tools import FileSearchTool

# One-time setup: create vector store and upload documents
client = OpenAI()
vs = client.beta.vector_stores.create(name="Company Docs")
client.beta.vector_stores.file_batches.upload_and_poll(
    vector_store_id=vs.id,
    files=[open("employee_handbook.pdf", "rb"), open("policies.pdf", "rb")]
)

agent = Agent(
    name="HR Assistant",
    instructions="Answer questions using company documents.",
    tools=[FileSearchTool(vector_store_ids=[vs.id])]
)

result = await Runner.run(agent, "What is our vacation policy?")
```

---

### ComputerTool

Lets the agent take screenshots, move the mouse, and type — real computer control.

```python
from agents.tools import ComputerTool

agent = Agent(
    name="Automation Agent",
    instructions="Automate computer tasks step by step.",
    tools=[ComputerTool(display=1, resolution=(1920, 1080))]
)
```

> ⚠️ **Use with caution.** This agent can interact with any application on your machine. Always run in a sandboxed VM with strong guardrails and human oversight.

---

### Mixing Hosted and Custom Tools

The agent automatically selects the right tool based on the user's request.

```python
agent = Agent(
    name="Enterprise Assistant",
    instructions="""
    Help users with information.
    - Use web search for current events.
    - Use file search for company policies.
    - Use query_database for order lookups.
    """,
    tools=[
        WebSearchTool(),
        FileSearchTool(vector_store_ids=[vs.id]),
        query_database,   # custom function
        send_email,       # custom function
    ]
)
```

| Tool | Risk | Setup | Best for |
|------|------|-------|----------|
| `WebSearchTool` | Low | None | Real-time web information |
| `FileSearchTool` | Low | Upload docs once | Document Q&A, knowledge bases |
| `ComputerTool` | **High** | Sandboxed VM | UI automation, testing |

**Docs:** https://openai.github.io/openai-agents-python/